In [9]:
# load “Imports + config” notebook
%run ./0_setup.ipynb

In [10]:
rasters = {
    "clay_0_30":             (SOIL_PATH, "Clay0-30cm.tif"),
    "sand_0_30":             (SOIL_PATH, "Sand0-30cm.tif"),
    "coarse_fragments_0_30": (SOIL_PATH, "CoarseFragments0-30cm.tif"),
    "bulk_density_0_30":     (SOIL_PATH, "BulkDensity0-30cm.tif"),
    "soc_0_30":              (SOIL_PATH, "SOC0-30cm.tif"),
    "ph_0_30":               (SOIL_PATH, "pH0-30cm.tif"),
    "cec_0_30":              (SOIL_PATH, "CEC0-30cm.tif"),
    "nitrogen_0_30":         (SOIL_PATH, "Nitrogen0-30cm.tif"),
    "elevation":             (GEO_PATH,     "elevation_1KMmn_SRTM.tiff"),
    "roughness":             (GEO_PATH,     "roughness_1KMmn_SRTM.tiff"),
    "slope":                 (GEO_PATH,     "slope_1KMmn_SRTM.tiff"),
    "tri":                   (GEO_PATH,     "tri_1KMmn_SRTM.tiff"),
    "rain_fall":             (CLIMATE_PATH, "CamRainFall.tif"),
    "temperature":           (CLIMATE_PATH, "CamTemp.tif"),
}

df = pd.read_csv(CSV_PATH)

vill_utm = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[LON_COL], df[LAT_COL]),
    crs="EPSG:32648"
)

# 1 km buffer (meters)
buffer_m = 1000
vill_buf_utm = vill_utm.copy()
vill_buf_utm["geometry"] = vill_utm.buffer(buffer_m)

for var, (folder, fname) in rasters.items():
    tif_path = os.path.join(folder, fname)

    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"Missing file for {var}: {tif_path}")

    print(f"Processing {var} -> {tif_path}")

    with rasterio.open(tif_path) as src:
        raster_crs = src.crs
        raster_nodata = src.nodata

    vill_buf = vill_buf_utm.to_crs(raster_crs)

    zs = zonal_stats(
        vill_buf,
        tif_path,
        stats=["mean"],
        nodata=raster_nodata,
        all_touched=False
    )

    vill_utm[f"{var}_mean_1km"] = [d["mean"] for d in zs]

print("All zonal statistics completed.")

# Save soil + geo + climate 
OUT_GEO_CLIMATE = BASE / "TM_Village_with_geo_climate_1km.csv"
vill_utm.drop(columns="geometry").to_csv(OUT_GEO_CLIMATE, index=False)
print("Saved:", OUT_GEO_CLIMATE)

Processing clay_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/Clay0-30cm.tif
Processing sand_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/Sand0-30cm.tif
Processing coarse_fragments_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/CoarseFragments0-30cm.tif
Processing bulk_density_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/BulkDensity0-30cm.tif
Processing soc_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/SOC0-30cm.tif
Processing ph_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/pH0-30cm.tif
Processing cec_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/CEC0-30cm.tif
Processing nitrogen_0_30 -> /Users/sam/Desktop/CERDIProject/TermiteMound/Script/R

In [11]:
vill_utm[[c for c in vill_utm.columns if "mean_1km" in c]].describe()

,clay_0_30_mean_1km,sand_0_30_mean_1km,coarse_fragments_0_30_mean_1km,bulk_density_0_30_mean_1km,soc_0_30_mean_1km,ph_0_30_mean_1km,cec_0_30_mean_1km,nitrogen_0_30_mean_1km,elevation_mean_1km,roughness_mean_1km,slope_mean_1km,tri_mean_1km,rain_fall_mean_1km,temperature_mean_1km
count,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000
mean,371.192272,306.881480,78.621169,139.144102,122.123127,58.155429,179.785044,153.720159,52.608551,6.081412,1.107085,2.049107,1497.635439,27.812635
std,32.985835,33.395873,19.546111,3.162278,22.275517,0.904544,9.472844,24.213055,33.677353,4.451836,0.995019,1.340571,107.824592,0.238044
min,302.040000,250.368062,53.897959,131.470000,87.826245,56.297872,156.812500,124.622449,14.750000,3.613333,0.560648,1.302917,1334.500000,26.779167
25%,343.780577,280.197098,61.998505,136.914747,105.260643,57.388298,173.505319,136.575133,26.321250,4.653333,0.771948,1.607708,1398.916667,27.769305
50%,370.326213,301.802083,74.234694,139.631192,118.578021,58.250000,180.288208,149.909729,39.213333,5.120000,0.892316,1.766250,1469.750000,27.890279
75%,397.464908,333.121150,89.843750,140.996951,136.897563,58.780433,186.406469,161.281175,72.802919,5.630000,1.036924,1.914969,1594.625000,27.959375
max,438.489583,384.064613,174.136680,145.549638,189.347229,60.510417,204.642375,240.020000,180.280014,44.709999,9.724940,13.657812,1715.500000,28.073959
